# Build CMS Hospital Analytics Database

## Objective

Load the validated processed CMS datasets into a SQLite relational database.

The database will contain three related tables:

- `hospitals` — one row per hospital
- `measures` — one row per CMS measure
- `hospital_measures` — one hospital-measure observation per reporting period

The database will support reproducible SQL analysis of hospital characteristics, emergency department performance, and CMS quality measures.

In [2]:
import sqlite3
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DB_PATH = PROJECT_ROOT / "data" / "hospital_quality.db"

print("Project root:", PROJECT_ROOT)
print("Processed data:", PROCESSED_DIR)
print("Database:", DB_PATH)

Project root: /Users/bonggonzalez/Documents/GitHub/cms-hospital-analytics
Processed data: /Users/bonggonzalez/Documents/GitHub/cms-hospital-analytics/data/processed
Database: /Users/bonggonzalez/Documents/GitHub/cms-hospital-analytics/data/hospital_quality.db


In [4]:
conn = sqlite3.connect(DB_PATH)

print("Database connected:", DB_PATH.exists())

Database connected: True


In [5]:
hospitals = pd.read_csv(PROCESSED_DIR / "hospitals.csv", dtype={"facility_id": str})
measures = pd.read_csv(PROCESSED_DIR / "measures.csv")
hospital_measures = pd.read_csv(
    PROCESSED_DIR / "hospital_measures.csv",
    dtype={"facility_id": str}
)

print("HOSPITALS")
print(hospitals.columns.tolist())

print("\nMEASURES")
print(measures.columns.tolist())

print("\nHOSPITAL_MEASURES")
print(hospital_measures.columns.tolist())

HOSPITALS
['facility_id', 'facility_name', 'address', 'city', 'state', 'zip_code', 'county', 'hospital_type', 'hospital_ownership', 'emergency_services', 'overall_rating', 'hospital_source', 'overall_rating_numeric']

MEASURES
['measure_id', 'measure_name', 'condition']

HOSPITAL_MEASURES
['facility_id', 'measure_id', 'score_raw', 'score_numeric', 'sample_raw', 'sample_numeric', 'footnote', 'start_date', 'end_date']


In [6]:
conn.execute("PRAGMA foreign_keys = ON;")

In [7]:
conn.execute("PRAGMA foreign_keys;").fetchone()

(1,)

In [8]:
conn.executescript("""
CREATE TABLE hospitals (
    facility_id TEXT PRIMARY KEY,
    facility_name TEXT NOT NULL,
    address TEXT,
    city TEXT,
    state TEXT,
    zip_code TEXT,
    county TEXT,
    hospital_type TEXT,
    hospital_ownership TEXT,
    emergency_services TEXT,
    overall_rating TEXT,
    hospital_source TEXT,
    overall_rating_numeric REAL
);

CREATE TABLE measures (
    measure_id TEXT PRIMARY KEY,
    measure_name TEXT NOT NULL,
    condition TEXT
);

CREATE TABLE hospital_measures (
    facility_id TEXT NOT NULL,
    measure_id TEXT NOT NULL,
    score_raw TEXT,
    score_numeric REAL,
    sample_raw TEXT,
    sample_numeric REAL,
    footnote TEXT,
    start_date TEXT NOT NULL,
    end_date TEXT NOT NULL,

    PRIMARY KEY (facility_id, measure_id, start_date, end_date),

    FOREIGN KEY (facility_id)
        REFERENCES hospitals(facility_id),

    FOREIGN KEY (measure_id)
        REFERENCES measures(measure_id)
);
""")

In [9]:
conn.execute("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
""").fetchall()

[('hospital_measures',), ('hospitals',), ('measures',)]

In [15]:
duplicate_ids.groupby("state").size().sort_values(ascending=False)

state
WI    240
TX    174
WY     56
GU      4
AS      2
MP      2
dtype: int64

In [16]:
duplicate_ids["facility_id"].nunique()

239